In [1]:
#  Imports & Config:

import os, json, torch, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Top-5 category mapping (1-indexed for Mask R-CNN, 0 = background)
TOP5_CATEGORIES = {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}  # note: 0 reserved for background
IDX_TO_NAME     = {1: 'short_sleeve_top', 2: 'trousers', 3: 'shorts',
                   4: 'long_sleeve_top',  5: 'skirt'}
NUM_CLASSES     = 6  # 5 classes + 1 background

DATA_ROOT  = '/kaggle/input/datasets/varun000reddy/'
TRAIN_IMG  = DATA_ROOT + 'training/train/image/'
TRAIN_ANN  = DATA_ROOT + 'training/train/annos/'
VAL_IMG    = DATA_ROOT + 'validation/validation/image/'
VAL_ANN    = DATA_ROOT + 'validation/validation/annos/'

SAVE_DIR   = '/kaggle/working/checkpoints/'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

Device : cuda
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128


In [2]:
# Dataset Class:

from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_convert
import skimage.draw

class ApparelMaskRCNNDataset(Dataset):
    def __init__(self, ann_dir, img_dir, max_samples=None):
        self.ann_dir  = ann_dir
        self.img_dir  = img_dir
        ann_files     = sorted([f for f in os.listdir(ann_dir) if f.endswith('.json')])

        # Filter to only images with top-5 categories
        self.valid_ids = []
        for fname in tqdm(ann_files[:max_samples] if max_samples else ann_files,
                          desc="Scanning annotations"):
            img_id   = fname.replace('.json', '')
            img_path = os.path.join(img_dir, img_id + '.jpg')
            if not os.path.exists(img_path):
                continue
            with open(os.path.join(ann_dir, fname)) as f:
                ann = json.load(f)
            has_top5 = any(
                item.get('category_id') in TOP5_CATEGORIES
                for key, item in ann.items()
                if key.startswith('item')
            )
            if has_top5:
                self.valid_ids.append(img_id)

        print(f"Found {len(self.valid_ids)} valid images")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        img_id   = self.valid_ids[idx]
        img_path = os.path.join(self.img_dir, img_id + '.jpg')
        ann_path = os.path.join(self.ann_dir, img_id + '.json')

        # Load image and resize
        img            = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size          # get original dimensions
        img            = img.resize((512, 512))
        img_w, img_h   = 512, 512
        img_tensor     = F.to_tensor(img)

        # Scale factors for coordinates
        scale_x = img_w / orig_w
        scale_y = img_h / orig_h

        with open(ann_path) as f:
            ann = json.load(f)

        boxes, labels, masks = [], [], []

        for key, item in ann.items():
            if not key.startswith('item'):
                continue
            cat_id = item.get('category_id')
            if cat_id not in TOP5_CATEGORIES:
                continue

            bbox = item.get('bounding_box', [])
            segs = item.get('segmentation', [])

            if not bbox or not segs:
                continue

            x1, y1, x2, y2 = bbox

            # Scale bbox to resized image
            x1 = x1 * scale_x
            y1 = y1 * scale_y
            x2 = x2 * scale_x
            y2 = y2 * scale_y

            # Clamp to image bounds
            x1 = max(0, min(x1, img_w - 1))
            y1 = max(0, min(y1, img_h - 1))
            x2 = max(0, min(x2, img_w - 1))
            y2 = max(0, min(y2, img_h - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(TOP5_CATEGORIES[cat_id])

            # Build binary mask from polygons
            mask = np.zeros((img_h, img_w), dtype=np.uint8)
            for polygon in segs:
                if len(polygon) < 6:
                    continue
                px = np.array(polygon[0::2], dtype=np.float32) * scale_x
                py = np.array(polygon[1::2], dtype=np.float32) * scale_y
                px = np.clip(px, 0, img_w - 1)
                py = np.clip(py, 0, img_h - 1)
                rr, cc = skimage.draw.polygon(py, px, shape=(img_h, img_w))
                mask[rr, cc] = 1
            masks.append(mask)

        if len(boxes) == 0:
            target = {
                'boxes'   : torch.zeros((0, 4), dtype=torch.float32),
                'labels'  : torch.zeros(0, dtype=torch.int64),
                'masks'   : torch.zeros((0, img_h, img_w), dtype=torch.uint8),
                'image_id': torch.tensor([idx])
            }
        else:
            target = {
                'boxes'   : torch.tensor(boxes,  dtype=torch.float32),
                'labels'  : torch.tensor(labels, dtype=torch.int64),
                'masks'   : torch.tensor(np.array(masks), dtype=torch.uint8),
                'image_id': torch.tensor([idx])
            }

        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


print("Building train dataset...")
train_dataset = ApparelMaskRCNNDataset(TRAIN_ANN, TRAIN_IMG, max_samples=5000)
print("Building val dataset...")
val_dataset   = ApparelMaskRCNNDataset(VAL_ANN,   VAL_IMG)

train_loader  = DataLoader(train_dataset, batch_size=6, shuffle=True,
                           num_workers=2, collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=6, shuffle=False,
                           num_workers=2, collate_fn=collate_fn)

print(f"\nTrain: {len(train_dataset)} | Val: {len(val_dataset)}")

Building train dataset...


Scanning annotations: 100%|██████████| 5000/5000 [00:42<00:00, 116.45it/s]


Found 3678 valid images
Building val dataset...


Scanning annotations: 100%|██████████| 32153/32153 [04:32<00:00, 117.88it/s]

Found 23741 valid images

Train: 3678 | Val: 23741


In [3]:

# Build Model:

def build_maskrcnn(pretrained=True):
    weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model   = maskrcnn_resnet50_fpn(weights=weights)

    # Replace box predictor
    in_features_box         = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, NUM_CLASSES)

    # Replace mask predictor
    in_features_mask        = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer            = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, NUM_CLASSES
    )
    return model

model = build_maskrcnn(pretrained=True).to(DEVICE)
print("Mask R-CNN loaded with pretrained ResNet-50 FPN backbone")
print(f"Number of classes: {NUM_CLASSES} (background + 5 apparel)")

for param in model.backbone.parameters():
    param.requires_grad = False
print("Backbone frozen for first 5 epochs")

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 196MB/s]


Mask R-CNN loaded with pretrained ResNet-50 FPN backbone
Number of classes: 6 (background + 5 apparel)
Backbone frozen for first 5 epochs


In [4]:
# Training Setup:

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.005, momentum=0.9, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
NUM_EPOCHS = 12        # changed from 10
UNFREEZE_EPOCH = 6     # added

In [5]:
# Training the model:

from tqdm import tqdm

# Copy checkpoint from saved dataset to working dir
import shutil
checkpoint_src = '/kaggle/input/datasets/pankajdeopa/vr-maskrcnn-checkpoint/checkpoints/maskrcnn_latest.pt'
checkpoint_dst = '/kaggle/working/checkpoints/maskrcnn_latest.pt'
os.makedirs('/kaggle/working/checkpoints/', exist_ok=True)

if os.path.exists(checkpoint_src):
    shutil.copy(checkpoint_src, checkpoint_dst)
    print("Checkpoint copied — will resume from saved epoch")
else:
    print("No checkpoint found — starting fresh")


# ── Resume Setup ───────────────────────────────────────────────
START_EPOCH = 1
best_loss   = float('inf')
resume_path = SAVE_DIR + 'maskrcnn_latest.pt'

if os.path.exists(resume_path):
    print("Found checkpoint — resuming training...")
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Skip optimizer state dict — incompatible due to frozen backbone
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    START_EPOCH = checkpoint['epoch'] + 1
    best_loss   = checkpoint['best_loss']
    print(f"Resumed from epoch {checkpoint['epoch']} | Best loss: {best_loss:.4f}")
else:
    print("No checkpoint found — starting fresh")


# ── Training Loop ──────────────────────────────────────────────
for epoch in range(START_EPOCH, NUM_EPOCHS + 1):

    # ── Unfreeze backbone at UNFREEZE_EPOCH ───────────────────
    if epoch == UNFREEZE_EPOCH:
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = torch.optim.SGD(
            [p for p in model.parameters() if p.requires_grad],
            lr=0.001, momentum=0.9, weight_decay=1e-4
        )
        print(f"Epoch {epoch}: Backbone unfrozen — fine-tuning all layers with LR=0.001")

    model.train()
    epoch_loss = 0.0
    train_bar  = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]")

    for imgs, targets in train_bar:
        imgs    = [img.to(DEVICE) for img in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += losses.item()
        train_bar.set_postfix(loss=f"{losses.item():.4f}")

    epoch_loss /= len(train_loader)
    scheduler.step()

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {epoch_loss:.4f}")

    # Save best model
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), SAVE_DIR + 'maskrcnn_best.pt')
        print(f"  ✓ Best model saved (Loss: {best_loss:.4f})")

    # Save latest checkpoint for resume
    torch.save({
        'epoch'               : epoch,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_loss'           : best_loss,
    }, SAVE_DIR + 'maskrcnn_latest.pt')
    print(f"  ✓ Checkpoint saved (epoch {epoch})")

print("\nTraining complete!")

Checkpoint copied — will resume from saved epoch
Found checkpoint — resuming training...
Resumed from epoch 2 | Best loss: 0.2768


Epoch 03/12 [Train]: 100%|██████████| 613/613 [08:02<00:00,  1.27it/s, loss=0.3025]


Epoch 03/12 | Loss: 0.2697
  ✓ Best model saved (Loss: 0.2697)
  ✓ Checkpoint saved (epoch 3)


Epoch 04/12 [Train]: 100%|██████████| 613/613 [08:10<00:00,  1.25it/s, loss=0.4000]


Epoch 04/12 | Loss: 0.2595
  ✓ Best model saved (Loss: 0.2595)
  ✓ Checkpoint saved (epoch 4)


...

Epoch 10/12 | Loss: 0.2108
  ✓ Best model saved (Loss: 0.2108)
  ✓ Checkpoint saved (epoch 10)


Epoch 11/12 [Train]: 100%|██████████| 613/613 [16:42<00:00,  1.64s/it, loss=0.1856]


Epoch 11/12 | Loss: 0.2060
  ✓ Best model saved (Loss: 0.2060)
  ✓ Checkpoint saved (epoch 11)


Epoch 12/12 [Train]: 100%|██████████| 613/613 [16:41<00:00,  1.63s/it, loss=0.2506]


Epoch 12/12 | Loss: 0.2004
  ✓ Best model saved (Loss: 0.2004)
  ✓ Checkpoint saved (epoch 12)

Training complete!


In [6]:
# Evaluation (mAP, mIoU, Dice):

from torchvision.ops import box_iou

def evaluate_maskrcnn(model, loader, iou_threshold=0.5):
    model.eval()
    all_ious, all_dice = [], []
    class_tp = {i: 0 for i in range(1, 6)}
    class_fp = {i: 0 for i in range(1, 6)}
    class_fn = {i: 0 for i in range(1, 6)}

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc='Evaluating'):
            imgs = [img.to(DEVICE) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes  = target['boxes'].to(DEVICE)
                gt_labels = target['labels'].to(DEVICE)
                gt_masks  = target['masks'].to(DEVICE).float()

                if len(pred['boxes']) == 0 or len(gt_boxes) == 0:
                    continue

                pred_boxes  = pred['boxes']
                pred_labels = pred['labels']
                pred_masks  = (pred['masks'][:, 0] > 0.5).float()
                pred_scores = pred['scores']

                # Filter by confidence
                keep        = pred_scores > 0.5
                pred_boxes  = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                pred_masks  = pred_masks[keep]

                if len(pred_boxes) == 0:
                    continue

                # Compute IoU between pred and gt boxes
                iou_matrix = box_iou(pred_boxes, gt_boxes)

                matched_gt = set()
                for p_idx in range(len(pred_boxes)):
                    best_iou, best_gt = iou_matrix[p_idx].max(0)
                    best_gt           = best_gt.item()
                    best_iou          = best_iou.item()
                    p_label           = pred_labels[p_idx].item()
                    g_label           = gt_labels[best_gt].item() if best_gt < len(gt_labels) else -1

                    if best_iou >= iou_threshold and p_label == g_label and best_gt not in matched_gt:
                        matched_gt.add(best_gt)
                        class_tp[p_label] = class_tp.get(p_label, 0) + 1

                        # Mask IoU and Dice
                        if best_gt < len(gt_masks) and p_idx < len(pred_masks):
                            pm = pred_masks[p_idx]
                            gm = gt_masks[best_gt]
                            if pm.shape == gm.shape:
                                intersection = (pm * gm).sum().item()
                                union        = (pm + gm).clamp(0, 1).sum().item()
                                mask_iou     = intersection / (union + 1e-6)
                                dice         = 2 * intersection / (pm.sum().item() + gm.sum().item() + 1e-6)
                                all_ious.append(mask_iou)
                                all_dice.append(dice)
                    else:
                        class_fp[p_label] = class_fp.get(p_label, 0) + 1

                for g_idx in range(len(gt_boxes)):
                    if g_idx not in matched_gt:
                        g_label = gt_labels[g_idx].item()
                        class_fn[g_label] = class_fn.get(g_label, 0) + 1

    # Compute per-class F1
    print("\n── Per-class Detection Results ──")
    print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-" * 45)
    macro_f1s = []
    for cls_idx in range(1, 6):
        tp = class_tp[cls_idx]
        fp = class_fp[cls_idx]
        fn = class_fn[cls_idx]
        p  = tp / (tp + fp + 1e-6)
        r  = tp / (tp + fn + 1e-6)
        f1 = 2 * p * r / (p + r + 1e-6)
        macro_f1s.append(f1)
        print(f"{IDX_TO_NAME[cls_idx]:<20} {p:>10.4f} {r:>10.4f} {f1:>10.4f}")

    print(f"\nMacro F1        : {np.mean(macro_f1s):.4f}")
    print(f"Mean mask IoU   : {np.mean(all_ious):.4f}")
    print(f"Mean Dice score : {np.mean(all_dice):.4f}")

    return np.mean(macro_f1s), np.mean(all_ious), np.mean(all_dice)

# Load best model and evaluate
model.load_state_dict(torch.load(SAVE_DIR + 'maskrcnn_best.pt', map_location=DEVICE))
macro_f1, miou, dice = evaluate_maskrcnn(model, val_loader)

Evaluating: 100%|██████████| 3957/3957 [42:21<00:00,  1.56it/s]


── Per-class Detection Results ──
Class                 Precision     Recall         F1
---------------------------------------------
short_sleeve_top         0.7487     0.9529     0.8385
trousers                 0.7732     0.9293     0.8441
shorts                   0.6377     0.8240     0.7190
long_sleeve_top          0.5916     0.8760     0.7062
skirt                    0.7064     0.8583     0.7750

Macro F1        : 0.7766
Mean mask IoU   : 0.8782
Mean Dice score : 0.9328


In [7]:
# Save to Kaggle Dataset:

import subprocess

os.makedirs('/kaggle/working/maskrcnn_upload/', exist_ok=True)

shutil.copy(SAVE_DIR + 'maskrcnn_best.pt',
            '/kaggle/working/maskrcnn_upload/maskrcnn_best.pt')

# Save metrics
import json
metrics = {'macro_f1': float(macro_f1), 'mIoU': float(miou), 'dice': float(dice)}
with open('/kaggle/working/maskrcnn_upload/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

metadata = {
    "title"   : "vr-maskrcnn-model",
    "id"      : "pankajdeopa/vr-maskrcnn-model",
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/kaggle/working/maskrcnn_upload/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', '/kaggle/working/maskrcnn_upload/', '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Starting upload for file maskrcnn_best.pt
Upload successful: maskrcnn_best.pt (168MB)
Starting upload for file metrics.json
Upload successful: metrics.json (96B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopa/vr-maskrcnn-model


  0%|          | 0.00/168M [00:00<?, ?B/s]
  6%|▋         | 10.7M/168M [00:00<00:01, 87.4MB/s]
 17%|█▋        | 29.4M/168M [00:00<00:01, 135MB/s] 
 25%|██▌       | 42.5M/168M [00:00<00:01, 119MB/s]
 32%|███▏      | 54.2M/168M [00:00<00:01, 93.1MB/s]
 41%|████      | 68.9M/168M [00:00<00:00, 108MB/s] 
 48%|████▊     | 80.1M/168M [00:00<00:00, 103MB/s]
 55%|█████▍    | 91.8M/168M [00:00<00:00, 93.3MB/s]
 65%|██████▌   | 109M/168M [00:01<00:00, 104MB/s]  
 71%|███████   | 120M/168M [00:01<00:00, 98.6MB/s]
 78%|███████▊  | 132M/168M [00:01<00:00, 96.7MB/s]
 89%|████████▊ | 149M/168M [00:01<00:00, 117MB/s] 
 96%|█████████▌| 161M/168M [00:01<00:00, 102MB/s]
100%|██████████| 168M/168M [00:02<00:00, 83.3MB/s]